# `check_on_less_data` (pre-flight methodology gate)
# DO NOT NEED TO RUN THIS IPYNB...ONLY  FOR TESTING

A **faithful miniature** of the whole pipeline on a label-stratified sub-sample, run **before** the
multi-hour main experiments (NB05/06/08). Same architecture, losses, toggles, uniform LR, and
script-aware BanglaBERT as the real notebooks — **only the data volume shrinks** and seeds are fixed
to one (the goal is a directional signal, not variance).

**Three legs + a verdict**
1. **Fine-tune** all 4 encoders (BanglishBERT / BanglaBERT / MuRIL / XLM-R), 1 seed → per-model 20%-test metrics + in-memory logits.
2. **Ensemble** — script-masked weighted (Nelder-Mead on val) → fused test metrics.
3. **Ablation** (BanglishBERT): additive component sweep + 5-vs-9-class taxonomy.
4. **Verdict** — PASS / REVISE. Flags: ensemble < best single, **any additive component with a
   non-positive Δ** (the "plus-delta" check), or 5-class not competitive with 9-class.

> If the verdict is **REVISE**, edit `plans.md §5` (drop/replace the offending component) and re-run
> this notebook until **PASS**, *then* launch the main experiments. Nothing here writes outside
> `outputs/precheck/`.

In [6]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [7]:
# ── (optional) install — uncomment if env differs ──
# !pip install torch==2.4.1 transformers==4.44.2 scikit-learn==1.5.1 pandas==2.2.2 numpy==1.26.4 sentencepiece==0.2.0 scipy==1.14.1 --quiet
print("deps assumed present")

deps assumed present


In [8]:
import os, json, time, math, random, re, warnings
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from scipy.optimize import minimize
from sklearn.metrics import f1_score, accuracy_score, matthews_corrcoef, roc_auc_score
warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA:", torch.cuda.is_available(),
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

CUDA: True | GPU: NVIDIA GeForce RTX 4060 Ti


In [9]:
# ═══════════════════════════════════════════════════════════════════
# CONFIG — IDENTICAL to NB05 (every technique on). Only data volume + seed differ.
# The whole point: keep settings the same so the signal transfers to the main run.
# ═══════════════════════════════════════════════════════════════════
# ── sub-sample size (the ONLY real lever) ──
N_TRAIN_PER_CLASS = 1500     # stratified on label5; min(n, available)
N_VAL_PER_CLASS   = 300
N_TEST_PER_CLASS  = 500
SEED              = 42        # one seed — this is a directional sanity check

CONFIG = {
    "models": {
        "banglishbert": "csebuetnlp/banglishbert",   # bilingual (Bangla + Romanized)
        "banglabert":   "csebuetnlp/banglabert",      # Bangla-script SPECIALIST (no Romanized)
        "muril":        "google/muril-base-cased",
        "xlmr":         "xlm-roberta-base",
    },
    "max_length": 128, "batch_size": 16, "eval_batch_size": 32, "grad_accum_steps": 2,
    "num_workers": 0, "epochs": 8, "encoder_lr": 2e-5, "head_lr": 8e-5,
    "weight_decay": 0.01, "warmup_ratio": 0.10, "max_grad_norm": 1.0,
    "label_smoothing": 0.03, "focal_gamma": 2.0, "class_weight_beta": 0.999,
    "dropout": 0.25, "head_hidden_dim": 384, "n_msd": 4, "patience": 3,
    "use_fp16": torch.cuda.is_available(),
    # ── advanced toggles (full method) ──
    "use_balanced_sampler": True, "sampler_alpha": 0.5,
    "per_class_loss_multiplier": {},
    "use_fgm":   True, "fgm_epsilon": 1.0,
    "use_rdrop": True, "rdrop_alpha": 1.0,
    "use_ema":   True, "ema_decay": 0.999,
    "banglabert_script_only": True,
    "banglabert_script_key": "banglabert",
}
USE_LRDECAY = False           # constraint 7 (uniform LR)

ABLATION_MODEL = "banglishbert"   # matches NB08 default; bilingual so it sees all data
LABEL_COL, TEXT_COL = "label5", "text_clean"
SPLIT_DIR = "../data/splits"
OUT = "../outputs/precheck"; os.makedirs(OUT, exist_ok=True)
RUN_MODELS = ["banglishbert", "banglabert", "muril", "xlmr"]

print(f"sample/class  train={N_TRAIN_PER_CLASS} val={N_VAL_PER_CLASS} test={N_TEST_PER_CLASS} | seed={SEED}")
print(f"toggles  sampler={CONFIG['use_balanced_sampler']} fgm={CONFIG['use_fgm']} "
      f"rdrop={CONFIG['use_rdrop']} ema={CONFIG['use_ema']} script_aware={CONFIG['banglabert_script_only']}")
print(f"models  {RUN_MODELS} | LRdecay={USE_LRDECAY} | ablation_anchor={ABLATION_MODEL}")

sample/class  train=1500 val=300 test=500 | seed=42
toggles  sampler=True fgm=True rdrop=True ema=True script_aware=True
models  ['banglishbert', 'banglabert', 'muril', 'xlmr'] | LRdecay=False | ablation_anchor=banglishbert


In [10]:
# ── Load full splits, then take a LABEL-STRATIFIED sub-sample ────────────────
# Stratify on label5 so every class is represented even in the miniature; cap each
# class at the per-split budget (or its full count if smaller). label_type/label_binary
# are kept for the 9-class taxonomy ablation; script is kept for BanglaBERT isolation.
# AFTER (pandas 3.0 compatible — iteration gives full groups including key column)
def stratified_sample(df, per_class, seed=SEED):
    parts = [
        g.sample(min(len(g), per_class), random_state=seed)
        for _, g in df.groupby(LABEL_COL)
    ]
    return pd.concat(parts).reset_index(drop=True)

_train_full = pd.read_csv(f"{SPLIT_DIR}/random_train.csv")
_val_full   = pd.read_csv(f"{SPLIT_DIR}/random_val.csv")
_test_full  = pd.read_csv(f"{SPLIT_DIR}/random_test.csv")

train_full = stratified_sample(_train_full, N_TRAIN_PER_CLASS)
val_df     = stratified_sample(_val_full,   N_VAL_PER_CLASS)
test_df    = stratified_sample(_test_full,  N_TEST_PER_CLASS)

classes = sorted(train_full[LABEL_COL].unique())
label_enc = {c: i for i, c in enumerate(classes)}
NUM_CLASSES = len(classes)
assert NUM_CLASSES == 5, f"expected 5 classes, got {NUM_CLASSES}: {classes}"

has_script = all("script" in d.columns for d in (train_full, val_df, test_df))
print(f"classes ({NUM_CLASSES}): {label_enc}")
print(f"sampled  train={len(train_full):,}  val={len(val_df):,}  test={len(test_df):,}")
print("\nper-class train counts:\n", train_full[LABEL_COL].value_counts().sort_index().to_string())
if has_script:
    print("\nscript mix (train):\n", train_full["script"].astype(str).str.lower().value_counts().to_string())
    bangla_n = train_full["script"].astype(str).str.lower().eq("bangla").sum()
    print(f"\n→ BanglaBERT will train on the {bangla_n:,} Bangla-script rows only.")
else:
    print("\n⚠ no 'script' column — BanglaBERT cannot be script-isolated; it will run full-scope.")

classes (5): {'abusive': 0, 'none': 1, 'religious': 2, 'sexual': 3, 'threat': 4}
sampled  train=7,500  val=1,500  test=2,500

per-class train counts:
 label5
abusive      1500
none         1500
religious    1500
sexual       1500
threat       1500

script mix (train):
 script
bangla       6009
romanized    1491

→ BanglaBERT will train on the 6,009 Bangla-script rows only.


In [11]:
# ── Dataset / Collator (identical to NB05) ──────────────────────────────────
class DS(Dataset):
    def __init__(self, df, tok, maxlen, enc, label=LABEL_COL):
        self.texts  = df.reset_index(drop=True)[TEXT_COL].fillna("").astype(str).tolist()
        self.labels = [int(enc.get(v, -1)) for v in df[label]]
        self.tok, self.maxlen = tok, maxlen
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        e = self.tok(self.texts[i], max_length=self.maxlen, truncation=True, padding=False)
        item = {k: torch.tensor(v, dtype=torch.long) for k, v in e.items()}
        item["label"] = torch.tensor(self.labels[i], dtype=torch.long)
        return item

class Collator:
    def __init__(self, tok): self.tok = tok
    def __call__(self, fs):
        labels = torch.stack([f["label"] for f in fs])
        feats = [{k: v for k, v in f.items() if k != "label"} for f in fs]
        b = self.tok.pad(feats, padding=True, return_tensors="pt"); b["label"] = labels
        return b
print("dataset ok")

dataset ok


In [12]:
# ── Loss, class weights, sampler (identical to NB05) ────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, label_smoothing=0.0):
        super().__init__(); self.gamma, self.weight, self.ls = gamma, weight, label_smoothing
    def forward(self, logits, t):
        ce = F.cross_entropy(logits, t, weight=self.weight, reduction="none", label_smoothing=self.ls)
        return (((1 - torch.exp(-ce)) ** self.gamma) * ce).mean()

def build_class_weights(series, enc, beta=0.999, max_w=10.0, per_class_mult=None):
    mapped = series.map(enc).dropna().astype(int); n = len(enc)
    counts = mapped.value_counts().sort_index(); w = np.ones(n, dtype=np.float32)
    for i in range(n):
        c = int(counts.get(i, 0))
        if c > 0: w[i] = 1.0 / max((1.0 - beta**c) / max(1.0 - beta, 1e-12), 1e-9)
    w = np.clip(w, w.min(), w.min() * max_w)
    if per_class_mult:
        for cname, m in per_class_mult.items():
            if cname in enc: w[enc[cname]] *= float(m)
    w = w / w.mean()
    return torch.tensor(w, dtype=torch.float32, device=device)

def make_balanced_sampler(df, enc, label=LABEL_COL, alpha=0.5):
    y = df[label].map(enc).fillna(-1).astype(int).values
    counts = np.bincount(y[y >= 0], minlength=len(enc)).astype(float); counts[counts == 0] = 1.0
    cw = (1.0 / counts) ** float(alpha)
    sw = np.where(y >= 0, cw[np.clip(y, 0, None)], 0.0)
    return WeightedRandomSampler(torch.as_tensor(sw, dtype=torch.double), len(sw), replacement=True)
print("loss/sampler ok")

loss/sampler ok


In [13]:
# ── Model: encoder + multi-sample-dropout head (identical to NB05) ──────────
class MSDHead(nn.Module):
    def __init__(self, hidden, n_cls, dropout=0.25, inner=384, n_msd=4):
        super().__init__(); inner = min(inner, hidden)
        self.pre = nn.Sequential(nn.Linear(hidden, inner), nn.GELU(), nn.LayerNorm(inner))
        self.drops = nn.ModuleList([nn.Dropout(dropout) for _ in range(max(1, n_msd))])
        self.out = nn.Linear(inner, n_cls)
    def forward(self, x):
        h = self.pre(x)
        if self.training and len(self.drops) > 1:
            return torch.stack([self.out(d(h)) for d in self.drops], 0).mean(0)
        return self.out(self.drops[0](h))

class Encoder(nn.Module):
    def __init__(self, name, n_cls, dropout=0.25, inner=384, n_msd=4):
        super().__init__(); self.encoder = AutoModel.from_pretrained(name)
        h = self.encoder.config.hidden_size
        self._tti = self.encoder.config.model_type.lower() not in ("roberta", "xlm-roberta")
        self.head = MSDHead(h, n_cls, dropout, inner, n_msd)
    def _pool(self, out, mask):
        hs = out.last_hidden_state; cls = hs[:, 0]
        mean = (hs * mask.unsqueeze(-1).float()).sum(1) / mask.sum(1, keepdim=True).float().clamp(1)
        return 0.5 * cls + 0.5 * mean
    def forward(self, ids, mask, tti=None):
        kw = {"input_ids": ids, "attention_mask": mask}
        if self._tti and tti is not None: kw["token_type_ids"] = tti
        return self.head(self._pool(self.encoder(**kw), mask))

def uniform_params(model, enc_lr, head_lr, wd):
    nd = ["bias", "LayerNorm.weight", "LayerNorm.bias"]; g = []
    for grp, lr in [(model.encoder, enc_lr), (model.head, head_lr)]:
        dec, ndec = [], []
        for n, p in grp.named_parameters():
            if p.requires_grad: (ndec if any(x in n for x in nd) else dec).append(p)
        g += [{"params": dec, "lr": lr, "weight_decay": wd},
              {"params": ndec, "lr": lr, "weight_decay": 0.0}]
    return g
print("model ok")

model ok


In [14]:
# ── FGM adversarial + EMA + R-Drop KL (identical to NB05) ───────────────────
class FGM:
    """Perturb token embeddings along the (normalized) gradient direction."""
    def __init__(self, model, epsilon=1.0, emb_key="word_embeddings"):
        self.model, self.eps, self.key, self.backup = model, epsilon, emb_key, {}
    def attack(self):
        for name, p in self.model.named_parameters():
            if p.requires_grad and self.key in name and p.grad is not None:
                self.backup[name] = p.data.clone()
                norm = torch.norm(p.grad)
                if norm != 0 and not torch.isnan(norm):
                    p.data.add_(self.eps * p.grad / norm)
    def restore(self):
        for name, p in self.model.named_parameters():
            if name in self.backup: p.data = self.backup[name]
        self.backup = {}

class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {n: p.detach().clone() for n, p in model.named_parameters() if p.requires_grad}
        self.backup = {}
    def update(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n].mul_(self.decay).add_(p.detach(), alpha=1 - self.decay)
    def apply_shadow(self, model):
        self.backup = {n: p.detach().clone() for n, p in model.named_parameters() if p.requires_grad}
        for n, p in model.named_parameters():
            if p.requires_grad: p.data.copy_(self.shadow[n])
    def restore(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad and n in self.backup: p.data.copy_(self.backup[n])
        self.backup = {}

def rdrop_kl(lp, lq):
    p_log, q_log = F.log_softmax(lp, -1), F.log_softmax(lq, -1)
    p, q = p_log.exp(), q_log.exp()
    return 0.5 * ((p * (p_log - q_log)).sum(-1) + (q * (q_log - p_log)).sum(-1)).mean()
print("fgm/ema/rdrop ok")

fgm/ema/rdrop ok


In [15]:
# ── Metrics + logits (identical to NB05) ────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader, nc=None):
    nc = nc or NUM_CLASSES
    model.eval(); P, Y, PR = [], [], []
    for b in loader:
        b = {k: v.to(device) for k, v in b.items()}
        lg = model(b["input_ids"], b["attention_mask"], b.get("token_type_ids"))
        pr = F.softmax(lg, -1).cpu().numpy(); y = b["label"].cpu().numpy(); m = y >= 0
        P.extend(pr[m].argmax(-1)); Y.extend(y[m]); PR.extend(pr[m])
    y, p, pr = np.array(Y), np.array(P), np.array(PR)
    rec = {"macro_f1": round(float(f1_score(y, p, average="macro", zero_division=0)), 4),
           "weighted_f1": round(float(f1_score(y, p, average="weighted", zero_division=0)), 4),
           "accuracy": round(float(accuracy_score(y, p)), 4),
           "mcc": round(float(matthews_corrcoef(y, p)), 4)}
    try:
        rec["auroc"] = round(float(roc_auc_score(y, pr, multi_class="ovr", average="macro",
                                                 labels=list(range(pr.shape[1])))), 4)
    except Exception:
        rec["auroc"] = float("nan")
    return rec

@torch.no_grad()
def get_logits(model, loader):
    model.eval(); out = []
    for b in loader:
        b = {k: v.to(device) for k, v in b.items()}
        out.append(model(b["input_ids"], b["attention_mask"], b.get("token_type_ids")).cpu())
    return torch.cat(out)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
print("metrics ok")

metrics ok


In [16]:
# ── Train one encoder (script-aware BanglaBERT; returns metrics + full-size logits) ─
# Same training machinery as NB05. BanglaBERT trains/evals on Bangla-only rows; its
# val/test logits are returned full-size [N, C] with zero rows for Romanized so the
# ensemble can mask them exactly like NB06.
def full_size_logits(model, subset_loader, full_n, subset_idx):
    raw = get_logits(model, subset_loader)
    if subset_idx is None:
        return raw
    full = torch.zeros((full_n, NUM_CLASSES), dtype=raw.dtype)
    full[torch.as_tensor(subset_idx, dtype=torch.long)] = raw
    return full

def train_one(model_key, model_name, seed=SEED, cfg=None):
    cfg = cfg or CONFIG
    set_seed(seed)
    sd = f"{OUT}/{model_key}_seed{seed}"; os.makedirs(sd, exist_ok=True)

    # ── Script-aware data scope: BanglaBERT = Bangla only ──
    is_script_model = (
        model_key == cfg.get("banglabert_script_key", "banglabert")
        and cfg.get("banglabert_script_only", False)
        and all("script" in d.columns for d in (train_full, val_df, test_df))
    )
    if is_script_model:
        bk = lambda d: d["script"].astype(str).str.lower().eq("bangla")
        train_df = train_full[bk(train_full)].reset_index(drop=True)
        val_eval_idx  = np.where(bk(val_df).values)[0]
        test_eval_idx = np.where(bk(test_df).values)[0]
        val_eval_df  = val_df.iloc[val_eval_idx].reset_index(drop=True)
        test_eval_df = test_df.iloc[test_eval_idx].reset_index(drop=True)
        for nm, d in [("train", train_df), ("val", val_eval_df), ("test", test_eval_df)]:
            if len(d) == 0: raise ValueError(f"{model_key}: no Bangla-script rows in {nm}.")
        print(f"  script-aware: {model_key} Bangla-only | train={len(train_df):,} "
              f"val={len(val_eval_df):,}/{len(val_df):,} test={len(test_eval_df):,}/{len(test_df):,}")
    else:
        train_df, val_eval_df, test_eval_df = train_full, val_df, test_df
        val_eval_idx = test_eval_idx = None

    tok = AutoTokenizer.from_pretrained(model_name); coll = Collator(tok)
    lkw = dict(collate_fn=coll, num_workers=cfg["num_workers"], pin_memory=torch.cuda.is_available())
    train_ds = DS(train_df, tok, cfg["max_length"], label_enc)
    if cfg["use_balanced_sampler"]:
        train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"],
                                  sampler=make_balanced_sampler(train_df, label_enc, LABEL_COL, cfg["sampler_alpha"]),
                                  shuffle=False, drop_last=True, **lkw)
    else:
        train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True, drop_last=True, **lkw)
    val_loader  = DataLoader(DS(val_eval_df,  tok, cfg["max_length"], label_enc), batch_size=cfg["eval_batch_size"], shuffle=False, **lkw)
    test_loader = DataLoader(DS(test_eval_df, tok, cfg["max_length"], label_enc), batch_size=cfg["eval_batch_size"], shuffle=False, **lkw)

    model = Encoder(model_name, NUM_CLASSES, cfg["dropout"], cfg["head_hidden_dim"], cfg["n_msd"]).to(device)
    cw = build_class_weights(train_df[LABEL_COL], label_enc, beta=cfg["class_weight_beta"],
                             per_class_mult=cfg.get("per_class_loss_multiplier", {}))
    focal = FocalLoss(cfg["focal_gamma"], cw, cfg["label_smoothing"])
    opt = torch.optim.AdamW(uniform_params(model, cfg["encoder_lr"], cfg["head_lr"], cfg["weight_decay"]))
    nsteps = math.ceil(len(train_loader) / cfg["grad_accum_steps"]) * cfg["epochs"]
    sch = get_linear_schedule_with_warmup(opt, int(nsteps * cfg["warmup_ratio"]), nsteps)
    scaler = torch.amp.GradScaler("cuda") if (cfg["use_fp16"] and device.type == "cuda") else None
    fgm = FGM(model, cfg["fgm_epsilon"]) if cfg["use_fgm"] else None
    ema = EMA(model, cfg["ema_decay"]) if cfg["use_ema"] else None
    best, noimp = -1.0, 0

    for ep in range(cfg["epochs"]):
        model.train(); opt.zero_grad(set_to_none=True); run, t0 = 0.0, time.time()
        for step, b in enumerate(train_loader, 1):
            b = {k: v.to(device) for k, v in b.items()}; y = b["label"]
            with torch.autocast(device_type=device.type, enabled=scaler is not None):
                lg1 = model(b["input_ids"], b["attention_mask"], b.get("token_type_ids"))
                if cfg["use_rdrop"]:
                    lg2 = model(b["input_ids"], b["attention_mask"], b.get("token_type_ids"))
                    loss = 0.5 * (focal(lg1, y) + focal(lg2, y)) + cfg["rdrop_alpha"] * rdrop_kl(lg1, lg2)
                else:
                    loss = focal(lg1, y)
                loss = loss / cfg["grad_accum_steps"]
            (scaler.scale(loss) if scaler else loss).backward()
            if fgm is not None:
                fgm.attack()
                with torch.autocast(device_type=device.type, enabled=scaler is not None):
                    la = focal(model(b["input_ids"], b["attention_mask"], b.get("token_type_ids")), y) / cfg["grad_accum_steps"]
                (scaler.scale(la) if scaler else la).backward(); fgm.restore()
            if step % cfg["grad_accum_steps"] == 0 or step == len(train_loader):
                if scaler: scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["max_grad_norm"])
                (scaler.step(opt), scaler.update()) if scaler else opt.step()
                sch.step(); opt.zero_grad(set_to_none=True)
                if ema is not None: ema.update(model)
            run += loss.item() * cfg["grad_accum_steps"]
        if ema is not None: ema.apply_shadow(model)
        vmac = evaluate(model, val_loader)["macro_f1"]; flag = ""
        if vmac > best:
            best, noimp = vmac, 0; torch.save(model.state_dict(), f"{sd}/best_model.pt"); flag = " ✅BEST"
        else: noimp += 1
        if ema is not None: ema.restore(model)
        print(f"  Ep{ep+1:2}/{cfg['epochs']} loss={run/max(len(train_loader),1):.4f} "
              f"val_macroF1={vmac:.4f} {(time.time()-t0)/60:.1f}min{flag}")
        if noimp >= cfg["patience"]:
            print(f"  early stop ep{ep+1}"); break

    model.load_state_dict(torch.load(f"{sd}/best_model.pt", map_location=device, weights_only=True))
    test_metrics = evaluate(model, test_loader)
    val_lg  = full_size_logits(model, val_loader,  len(val_df),  val_eval_idx)
    test_lg = full_size_logits(model, test_loader, len(test_df), test_eval_idx)
    res = {"model": model_key, "seed": seed,
           "eval_scope": "bangla_only" if is_script_model else "full",
           "best_val_macro_f1": round(best, 4), "test_metrics": test_metrics}
    json.dump(res, open(f"{sd}/results.json", "w"), indent=2)
    scope = "Bangla-only 20% TEST" if is_script_model else "20% TEST"
    print(f"  → [{scope}] macroF1={test_metrics['macro_f1']:.4f} wF1={test_metrics['weighted_f1']:.4f} "
          f"acc={test_metrics['accuracy']:.4f} mcc={test_metrics['mcc']:.4f}")
    del model; torch.cuda.empty_cache()
    return res, val_lg, test_lg
print("trainer ok")

trainer ok


## Leg A — Fine-tune the 4 encoders (1 seed)

Trains BanglishBERT / BanglaBERT / MuRIL / XLM-R on the sub-sample with the full method. BanglaBERT
runs Bangla-only; the others see both scripts. Logits are kept in memory for the ensemble.

In [17]:
# ── Run the 4 encoders ──────────────────────────────────────────────────────
finetune_results, VAL_LOGITS, TEST_LOGITS = [], {}, {}
t_all = time.time()
for mk in RUN_MODELS:
    print(f"\n▶ {mk}")
    try:
        res, vlg, tlg = train_one(mk, CONFIG["models"][mk])
        finetune_results.append(res)
        VAL_LOGITS[f"{mk}_seed{SEED}"]  = vlg
        TEST_LOGITS[f"{mk}_seed{SEED}"] = tlg
    except Exception as e:
        import traceback; print(f"❌ {mk}: {e}"); traceback.print_exc()
    torch.cuda.empty_cache()
print(f"\n🎉 {len(finetune_results)}/{len(RUN_MODELS)} encoders done in {(time.time()-t_all)/60:.1f} min")


▶ banglishbert


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraModel LOAD REPORT from: csebuetnlp/banglishbert
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Ep 1/8 loss=0.7430 val_macroF1=0.1538 1.3min ✅BEST
  Ep 2/8 loss=0.3695 val_macroF1=0.3579 1.2min ✅BEST
  Ep 3/8 loss=0.2606 val_macroF1=0.5570 1.2min ✅BEST
  Ep 4/8 loss=0.1918 val_macroF1=0.6785 1.2min ✅BEST
  Ep 5/8 loss=0.1319 val_macroF1=0.7144 1.2min ✅BEST
  Ep 6/8 loss=0.1063 val_macroF1=0.7259 1.2min ✅BEST
  Ep 7/8 loss=0.0851 val_macroF1=0.7349 1.4min ✅BEST
  Ep 8/8 loss=0.0675 val_macroF1=0.7427 1.7min ✅BEST
  → [20% TEST] macroF1=0.7524 wF1=0.7524 acc=0.7520 mcc=0.6901

▶ banglabert
  script-aware: banglabert Bangla-only | train=6,009 val=1,219/1,500 test=1,983/2,500


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

[transformers] BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Ep 1/8 loss=0.8631 val_macroF1=0.0667 3.0min ✅BEST
  Ep 2/8 loss=0.4145 val_macroF1=0.0667 5.2min
  Ep 3/8 loss=0.3403 val_macroF1=0.0969 5.1min ✅BEST
  Ep 4/8 loss=0.2758 val_macroF1=0.3591 7.9min ✅BEST
  Ep 5/8 loss=0.2194 val_macroF1=0.4825 10.1min ✅BEST
  Ep 6/8 loss=0.1947 val_macroF1=0.6202 11.2min ✅BEST
  Ep 7/8 loss=0.1724 val_macroF1=0.6966 10.9min ✅BEST
  Ep 8/8 loss=0.1510 val_macroF1=0.7275 9.7min ✅BEST
  → [20% TEST] macroF1=0.7370 wF1=0.7370 acc=0.7380 mcc=0.6786

▶ xlmr


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Ep 1/8 loss=0.7775 val_macroF1=0.0800 12.8min ✅BEST
  Ep 2/8 loss=0.4211 val_macroF1=0.2103 14.9min ✅BEST
  Ep 3/8 loss=0.3301 val_macroF1=0.4457 16.7min ✅BEST
  Ep 4/8 loss=0.2659 val_macroF1=0.5721 16.6min ✅BEST
  Ep 5/8 loss=0.2104 val_macroF1=0.6436 18.6min ✅BEST
  Ep 6/8 loss=0.1875 val_macroF1=0.7023 20.3min ✅BEST
  Ep 7/8 loss=0.1583 val_macroF1=0.7130 19.1min ✅BEST
  Ep 8/8 loss=0.1323 val_macroF1=0.7128 16.7min
  → [20% TEST] macroF1=0.7283 wF1=0.7283 acc=0.7356 mcc=0.6737

🎉 4/4 encoders done in 219.7 min


In [18]:
# ── Per-encoder test summary ────────────────────────────────────────────────
ft_df = pd.DataFrame([{"model": r["model"], "scope": r["eval_scope"],
                       "best_val_mF1": r["best_val_macro_f1"], **r["test_metrics"]}
                      for r in finetune_results])
print(ft_df.to_string(index=False))
ft_df.to_csv(f"{OUT}/finetune_summary.csv", index=False)
best_single = float(ft_df["macro_f1"].max()) if len(ft_df) else float("nan")
print(f"\nbest single-encoder macro-F1 = {best_single:.4f}")

       model       scope  best_val_mF1  macro_f1  weighted_f1  accuracy    mcc  auroc
banglishbert        full        0.7427    0.7524       0.7524    0.7520 0.6901 0.9400
  banglabert bangla_only        0.6795    0.7063       0.7341    0.7252 0.6546 0.9273
       muril        full        0.7275    0.7370       0.7370    0.7380 0.6786 0.9393
        xlmr        full        0.7130    0.7283       0.7283    0.7356 0.6737 0.9352

best single-encoder macro-F1 = 0.7524


## Leg B — Script-masked weighted ensemble

Fuses the 4 encoders' logits with weights optimised on the **val** split (Nelder-Mead, multi-restart),
exactly mirroring NB06: **BanglaBERT is masked to zero on Romanized rows** and weights renormalise
over the active encoders per row. Reported on the pure 20% test.

In [19]:
# ── Script-aware masks (mirror NB06) ────────────────────────────────────────
def is_banglabert_run(name): return name.startswith("banglabert_seed")  # not banglishbert

if has_script:
    val_is_bangla  = torch.tensor(val_df["script"].astype(str).str.lower().eq("bangla").values,  dtype=torch.bool)
    test_is_bangla = torch.tensor(test_df["script"].astype(str).str.lower().eq("bangla").values, dtype=torch.bool)
else:
    val_is_bangla  = torch.ones(len(val_df),  dtype=torch.bool)
    test_is_bangla = torch.ones(len(test_df), dtype=torch.bool)

names = list(VAL_LOGITS.keys())
val_active  = {n: (val_is_bangla  if is_banglabert_run(n) else torch.ones(len(val_df),  dtype=torch.bool)) for n in names}
test_active = {n: (test_is_bangla if is_banglabert_run(n) else torch.ones(len(test_df), dtype=torch.bool)) for n in names}
# safety: force neutral logits on inactive rows
for n in names:
    VAL_LOGITS[n]  = VAL_LOGITS[n].clone();  VAL_LOGITS[n][~val_active[n]]  = 0.0
    TEST_LOGITS[n] = TEST_LOGITS[n].clone(); TEST_LOGITS[n][~test_active[n]] = 0.0
print("active encoders per split ready:", names)

active encoders per split ready: ['banglishbert_seed42', 'banglabert_seed42', 'muril_seed42', 'xlmr_seed42']


In [20]:
# ── Masked weighted fusion + Nelder-Mead weight search on val ───────────────
y_val  = val_df[LABEL_COL].map(label_enc).values
y_test = test_df[LABEL_COL].map(label_enc).values

def masked_ensemble(logits_d, active_d, w):
    num = torch.zeros_like(next(iter(logits_d.values())))
    den = torch.zeros(num.shape[0], 1)
    for i, n in enumerate(names):
        a = active_d[n].float().unsqueeze(1)
        num = num + w[i] * logits_d[n] * a
        den = den + w[i] * a
    return num / (den + 1e-12)

def opt_weights(logits_d, active_d, y, k):
    best, bw = 1.0, np.ones(k) / k
    for _ in range(30):
        r = minimize(lambda rw: -f1_score(y, masked_ensemble(logits_d, active_d, np.abs(rw)+1e-9).argmax(-1).numpy(),
                                          average="macro", zero_division=0),
                     np.random.dirichlet(np.ones(k)), method="Nelder-Mead",
                     options={"maxiter": 1500, "xatol": 1e-5})
        if r.fun < best: best, bw = r.fun, np.abs(r.x) + 1e-9
    return bw / bw.sum()

W = opt_weights(VAL_LOGITS, val_active, y_val, len(names))
ens_test = masked_ensemble(TEST_LOGITS, test_active, W)
yp = ens_test.argmax(-1).numpy(); pr = F.softmax(ens_test, -1).numpy()
ens_metrics = {"macro_f1": round(float(f1_score(y_test, yp, average="macro", zero_division=0)), 4),
               "weighted_f1": round(float(f1_score(y_test, yp, average="weighted", zero_division=0)), 4),
               "accuracy": round(float(accuracy_score(y_test, yp)), 4),
               "mcc": round(float(matthews_corrcoef(y_test, yp)), 4)}
try: ens_metrics["auroc"] = round(float(roc_auc_score(y_test, pr, multi_class="ovr", average="macro",
                                                       labels=list(range(NUM_CLASSES)))), 4)
except Exception: ens_metrics["auroc"] = float("nan")
print("ensemble weights:", {n: round(float(w), 3) for n, w in zip(names, W)})
print("ENSEMBLE (20% test):", ens_metrics)
json.dump({"weights": {n: float(w) for n, w in zip(names, W)}, "metrics": ens_metrics},
          open(f"{OUT}/ensemble_metrics.json", "w"), indent=2)

ensemble weights: {'banglishbert_seed42': 0.488, 'banglabert_seed42': 0.025, 'muril_seed42': 0.392, 'xlmr_seed42': 0.095}
ENSEMBLE (20% test): {'macro_f1': 0.7649, 'weighted_f1': 0.7649, 'accuracy': 0.764, 'mcc': 0.7054, 'auroc': 0.9459}


## Leg C — Component ablation (additive, BanglishBERT)

Mirrors NB08: starting from a plain CE baseline, add one component at a time and read the macro-F1
**step delta**. Same architecture/LR/data as above; only the toggles flip. A non-positive step delta
is the signal that a component isn't earning its place.

In [21]:
# ── Ablation runner (BanglishBERT; CE→focal switchable, like NB08) ──────────
ABL_PATH = CONFIG["models"][ABLATION_MODEL]

def run_config(cfg, label_col, tr_df, va_df, te_df, enc, nc, tag):
    set_seed(SEED)
    tok = AutoTokenizer.from_pretrained(ABL_PATH); coll = Collator(tok)
    lkw = dict(collate_fn=coll, num_workers=cfg["num_workers"], pin_memory=torch.cuda.is_available())
    ds = DS(tr_df, tok, cfg["max_length"], enc, label_col)
    if cfg["use_balanced_sampler"]:
        tl = DataLoader(ds, batch_size=cfg["batch_size"],
                        sampler=make_balanced_sampler(tr_df, enc, label_col, cfg["sampler_alpha"]),
                        shuffle=False, drop_last=True, **lkw)
    else:
        tl = DataLoader(ds, batch_size=cfg["batch_size"], shuffle=True, drop_last=True, **lkw)
    vl = DataLoader(DS(va_df, tok, cfg["max_length"], enc, label_col), batch_size=cfg["eval_batch_size"], shuffle=False, **lkw)
    te = DataLoader(DS(te_df, tok, cfg["max_length"], enc, label_col), batch_size=cfg["eval_batch_size"], shuffle=False, **lkw)

    model = Encoder(ABL_PATH, nc, cfg["dropout"], cfg["head_hidden_dim"], cfg["n_msd"]).to(device)
    cw = build_class_weights(tr_df[label_col], enc, cfg["class_weight_beta"]) if cfg["use_cw"] else None
    crit = FocalLoss(cfg["focal_gamma"], cw, cfg["label_smoothing"]) if cfg["use_focal"] \
           else (lambda lg, t: F.cross_entropy(lg, t, weight=cw, label_smoothing=cfg["label_smoothing"]))
    opt = torch.optim.AdamW(uniform_params(model, cfg["encoder_lr"], cfg["head_lr"], cfg["weight_decay"]))
    ns = math.ceil(len(tl)/cfg["grad_accum_steps"]) * cfg["epochs"]
    sch = get_linear_schedule_with_warmup(opt, int(ns*cfg["warmup_ratio"]), ns)
    scaler = torch.amp.GradScaler("cuda") if (cfg["use_fp16"] and device.type == "cuda") else None
    fgm = FGM(model, cfg["fgm_epsilon"]) if cfg["use_fgm"] else None
    ema = EMA(model, cfg["ema_decay"]) if cfg["use_ema"] else None
    best, noimp, t0 = -1.0, 0, time.time()
    for ep in range(cfg["epochs"]):
        model.train(); opt.zero_grad(set_to_none=True)
        for step, b in enumerate(tl, 1):
            b = {k: v.to(device) for k, v in b.items()}; y = b["label"]
            with torch.autocast(device_type=device.type, enabled=scaler is not None):
                l1 = model(b["input_ids"], b["attention_mask"], b.get("token_type_ids"))
                if cfg["use_rdrop"]:
                    l2 = model(b["input_ids"], b["attention_mask"], b.get("token_type_ids"))
                    loss = 0.5*(crit(l1, y) + crit(l2, y)) + cfg["rdrop_alpha"]*rdrop_kl(l1, l2)
                else: loss = crit(l1, y)
                loss = loss / cfg["grad_accum_steps"]
            (scaler.scale(loss) if scaler else loss).backward()
            if fgm is not None:
                fgm.attack()
                with torch.autocast(device_type=device.type, enabled=scaler is not None):
                    la = crit(model(b["input_ids"], b["attention_mask"], b.get("token_type_ids")), y) / cfg["grad_accum_steps"]
                (scaler.scale(la) if scaler else la).backward(); fgm.restore()
            if step % cfg["grad_accum_steps"] == 0 or step == len(tl):
                if scaler: scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["max_grad_norm"])
                (scaler.step(opt), scaler.update()) if scaler else opt.step()
                sch.step(); opt.zero_grad(set_to_none=True)
                if ema is not None: ema.update(model)
        if ema is not None: ema.apply_shadow(model)
        vm = evaluate(model, vl, nc)["macro_f1"]
        if vm > best: best, noimp = vm, 0; torch.save(model.state_dict(), f"{OUT}/_tmp_best.pt")
        else: noimp += 1
        if ema is not None: ema.restore(model)
        if noimp >= cfg["patience"]: break
    model.load_state_dict(torch.load(f"{OUT}/_tmp_best.pt", map_location=device, weights_only=True))
    m = evaluate(model, te, nc); m["config"] = tag; m["minutes"] = round((time.time()-t0)/60, 1)
    del model; torch.cuda.empty_cache()
    print(f"  {tag:14s} macroF1={m['macro_f1']} wF1={m['weighted_f1']} acc={m['accuracy']} mcc={m['mcc']} [{m['minutes']}min]")
    return m
print("ablation runner ok")

ablation runner ok


In [22]:
# ── A. Component ablation (additive) ────────────────────────────────────────
# BASE = plain CE, no sampler/MSD/RDrop/FGM/EMA. Each step turns ONE thing on and stays on.
ABL_BASE = {**CONFIG, "use_focal": False, "use_cw": False, "use_balanced_sampler": False,
            "n_msd": 1, "use_rdrop": False, "use_fgm": False, "use_ema": False}
steps = [("base",        {}),
         ("+focal+CW",   {"use_focal": True, "use_cw": True}),
         ("+sampler",    {"use_balanced_sampler": True}),
         ("+MSD",        {"n_msd": 4}),
         ("+RDrop",      {"use_rdrop": True}),
         ("+FGM",        {"use_fgm": True}),
         ("+EMA(full)",  {"use_ema": True})]

cfg = dict(ABL_BASE); component = []
print(f"A. COMPONENT ABLATION (5-class, {ABLATION_MODEL})")
for name, delta in steps:
    cfg.update(delta)
    component.append(run_config(dict(cfg), LABEL_COL, train_full, val_df, test_df, label_enc, NUM_CLASSES, name))
comp_df = pd.DataFrame(component)
comp_df["delta_macro_f1"] = comp_df["macro_f1"].diff().round(4)   # step-wise contribution
comp_df.to_csv(f"{OUT}/component_ablation.csv", index=False)
print("\n", comp_df[["config", "macro_f1", "delta_macro_f1", "weighted_f1", "accuracy", "mcc"]].to_string(index=False))
FULL_5CLASS = component[-1]   # reuse for the taxonomy leg

A. COMPONENT ABLATION (5-class, banglishbert)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraModel LOAD REPORT from: csebuetnlp/banglishbert
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  base           macroF1=0.7643 wF1=0.7643 acc=0.7648 mcc=0.7062 [4.8min]


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

[transformers] ElectraModel LOAD REPORT from: csebuetnlp/banglishbert
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  +FGM           macroF1=0.7667 wF1=0.7667 acc=0.768 mcc=0.7103 [10.2min]


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraModel LOAD REPORT from: csebuetnlp/banglishbert
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  +EMA(full)     macroF1=0.7512 wF1=0.7512 acc=0.7508 mcc=0.6886 [9.7min]

     config  macro_f1  delta_macro_f1  weighted_f1  accuracy    mcc
      base    0.7643             NaN       0.7643    0.7648 0.7062
 +focal+CW    0.7518         -0.0125       0.7518    0.7540 0.6930
  +sampler    0.7440         -0.0078       0.7440    0.7448 0.6811
      +MSD    0.7389         -0.0051       0.7389    0.7400 0.6757
    +RDrop    0.7356         -0.0033       0.7356    0.7332 0.6673
      +FGM    0.7667          0.0311       0.7667    0.7680 0.7103
+EMA(full)    0.7512         -0.0155       0.7512    0.7508 0.6886


## Leg D — Taxonomy ablation (5-class vs 9-class)

Re-derives the old 9-class labels from `label_type` (+ `label_binary`) on the same rows, then trains
the full method on each. *Note:* on a small stratified sample the rare 9-classes (gender, political)
may be tiny, so treat this as directional. The 5-class full run is reused from Leg C.

In [23]:
# ── B. Taxonomy ablation: 5-class (reused) vs 9-class (full method) ─────────
NINE_MAP = {"none":"none","not bully":"none","sexual":"sexual","gender":"gender","religious":"religious",
            "religion":"religious","threat":"threat","calltoviolence":"threat","abusive/violence":"abusive",
            "abusive":"abusive","violence":"abusive","personal offense":"personal","personal":"personal",
            "political":"political","troll":"abusive","slander":"other","spam":"other","origin":"other",
            "body shaming":"other","misc":"other"}
NINE_PRIO = ["threat","sexual","religious","gender","political","abusive","personal","other","none"]
def to9(raw):
    if not isinstance(raw, str) or not raw.strip(): return "none"
    parts = [p.strip().lower() for p in re.split(r"[,_]", raw.strip()) if p.strip()]; b = set()
    for p in parts:
        if p in NINE_MAP: b.add(NINE_MAP[p])
        else:
            for k, v in NINE_MAP.items():
                if k in p: b.add(v); break
    if not b: return "other"
    for c in NINE_PRIO:
        if c in b: return c
    return "none"

have9 = all(c in train_full.columns for c in ("label_type", "label_binary"))
tax = [{**FULL_5CLASS, "config": "5-class(full)"}]
if have9:
    for d in (train_full, val_df, test_df):
        d["label9"] = d["label_type"].apply(to9)
        d.loc[d["label_binary"] == 0, "label9"] = "none"
    classes9 = sorted(set(train_full["label9"]) | set(val_df["label9"]) | set(test_df["label9"]))
    enc9 = {c: i for i, c in enumerate(classes9)}
    full = {**CONFIG, "use_focal": True, "use_cw": True, "use_balanced_sampler": True,
            "n_msd": 4, "use_rdrop": True, "use_fgm": True, "use_ema": True}
    print(f"B. TAXONOMY ABLATION (full method) | 9-class = {classes9}")
    tax.append(run_config(dict(full), "label9", train_full, val_df, test_df, enc9, len(classes9), "9-class(full)"))
else:
    print("⚠ label_type/label_binary missing — skipping 9-class derivation.")
tax_df = pd.DataFrame(tax); tax_df.to_csv(f"{OUT}/taxonomy_ablation.csv", index=False)
print("\n", tax_df[["config", "macro_f1", "weighted_f1", "accuracy", "mcc"]].to_string(index=False))
print("\nNote: macro-F1 across different class counts is not strictly comparable; read alongside")
print("per-class support. The 5-class case is justified if it is at least competitive and its rare")
print("classes are better-populated than the 9-class equivalents.")

B. TAXONOMY ABLATION (full method) | 9-class = ['abusive', 'gender', 'none', 'other', 'personal', 'political', 'religious', 'sexual', 'threat']


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraModel LOAD REPORT from: csebuetnlp/banglishbert
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  9-class(full)  macroF1=0.4345 wF1=0.6185 acc=0.6116 mcc=0.5473 [10.1min]

        config  macro_f1  weighted_f1  accuracy    mcc
5-class(full)    0.7512       0.7512    0.7508 0.6886
9-class(full)    0.4345       0.6185    0.6116 0.5473

Note: macro-F1 across different class counts is not strictly comparable; read alongside
per-class support. The 5-class case is justified if it is at least competitive and its rare
classes are better-populated than the 9-class equivalents.


## Verdict — PASS / REVISE

Automated read of the three legs. **REVISE** if the ensemble underperforms the best single encoder,
if any additive component contributes a non-positive macro-F1 delta (the "plus-delta" check), or if
5-class clearly trails 9-class. Otherwise **PASS** → launch the main experiments.

In [24]:
# ── Verdict ─────────────────────────────────────────────────────────────────
TOL = 0.003   # tolerance band (~0.3 macro-F1 pts) — small-sample noise, not a real regression

flags = []

# 1) ensemble should not lose to the best single encoder
ens_mf1 = ens_metrics["macro_f1"]
if ens_mf1 < best_single - TOL:
    flags.append(f"ensemble ({ens_mf1:.4f}) < best single ({best_single:.4f}) by >{TOL}")

# 2) every additive component must contribute a non-negative step delta
harmful = comp_df.dropna(subset=["delta_macro_f1"])
harmful = harmful[harmful["delta_macro_f1"] < -TOL]
for _, r in harmful.iterrows():
    flags.append(f"component '{r['config']}' hurts: ΔmacroF1={r['delta_macro_f1']:+.4f} → revise/drop it")

# 3) full method should be the best (or tied) configuration in the additive sweep
full_mf1 = comp_df["macro_f1"].iloc[-1]; peak_mf1 = comp_df["macro_f1"].max()
if full_mf1 < peak_mf1 - TOL:
    peak_cfg = comp_df.loc[comp_df["macro_f1"].idxmax(), "config"]
    flags.append(f"full method ({full_mf1:.4f}) below peak config '{peak_cfg}' ({peak_mf1:.4f}) → trim to the peak")

# 4) taxonomy: 5-class should be competitive with 9-class (informational threshold)
if len(tax_df) == 2:
    m5 = tax_df.loc[tax_df["config"]=="5-class(full)", "macro_f1"].iloc[0]
    m9 = tax_df.loc[tax_df["config"]=="9-class(full)", "macro_f1"].iloc[0]
    if m5 < m9 - 0.02:
        flags.append(f"5-class ({m5:.4f}) trails 9-class ({m9:.4f}) by >0.02 — re-examine taxonomy (directional)")

print("="*64)
print("PRE-FLIGHT VERDICT")
print("="*64)
print(f"  best single encoder macroF1 : {best_single:.4f}")
print(f"  ensemble macroF1            : {ens_mf1:.4f}")
print(f"  full-method (ablation) mF1  : {full_mf1:.4f}  (peak in sweep: {peak_mf1:.4f})")
print(f"  component step deltas       : "
      + ", ".join(f"{r['config']}={r['delta_macro_f1']:+.3f}"
                  for _, r in comp_df.dropna(subset=['delta_macro_f1']).iterrows()))
print("-"*64)
if not flags:
    print("  ✅ PASS — methodology behaves as designed. Clear to launch NB05/06/07/08.")
else:
    print("  ⚠ REVISE — address before committing GPU-days:")
    for f in flags: print(f"     • {f}")
print("="*64)
json.dump({"pass": len(flags)==0, "flags": flags,
           "best_single": best_single, "ensemble": ens_metrics,
           "component_ablation": comp_df.to_dict("records"),
           "taxonomy": tax_df.to_dict("records")},
          open(f"{OUT}/verdict.json", "w"), indent=2, default=float)
print(f"\nsaved → {OUT}/verdict.json (+ finetune_summary, ensemble_metrics, component_ablation, taxonomy_ablation CSVs)")

PRE-FLIGHT VERDICT
  best single encoder macroF1 : 0.7524
  ensemble macroF1            : 0.7649
  full-method (ablation) mF1  : 0.7512  (peak in sweep: 0.7667)
  component step deltas       : +focal+CW=-0.013, +sampler=-0.008, +MSD=-0.005, +RDrop=-0.003, +FGM=+0.031, +EMA(full)=-0.015
----------------------------------------------------------------
  ⚠ REVISE — address before committing GPU-days:
     • component '+focal+CW' hurts: ΔmacroF1=-0.0125 → revise/drop it
     • component '+sampler' hurts: ΔmacroF1=-0.0078 → revise/drop it
     • component '+MSD' hurts: ΔmacroF1=-0.0051 → revise/drop it
     • component '+RDrop' hurts: ΔmacroF1=-0.0033 → revise/drop it
     • component '+EMA(full)' hurts: ΔmacroF1=-0.0155 → revise/drop it
     • full method (0.7512) below peak config '+FGM' (0.7667) → trim to the peak

saved → ../outputs/precheck/verdict.json (+ finetune_summary, ensemble_metrics, component_ablation, taxonomy_ablation CSVs)


---
**Outputs** (all under `outputs/precheck/`): `finetune_summary.csv`, `ensemble_metrics.json`,
`component_ablation.csv`, `taxonomy_ablation.csv`, `verdict.json`, per-encoder `results.json`.

This notebook is a gate, not a result: its numbers come from a tiny sample and are **not** reported in
the paper. It only answers *"is the method pointed the right way before I spend the GPU-days?"*. If
PASS, run the real pipeline; if REVISE, edit `plans.md §5` and re-run.